# Data Engineering Foundations (Modules 1–4) — Merged Notes

This notebook consolidates and improves notes from:
- 1: Data Engineering Foundations
- 2: Google Cloud Storage and BigQuery
- 3: BigQuery hands-on
- 4: Pipeline types and orchestration

## Learning Goals
By the end of this notebook, you should be able to:
1. Explain the data engineering ecosystem and lifecycle.
2. Distinguish Cloud Storage, Datastore/Firestore, and BigQuery use cases.
3. Load local CSV data into BigQuery and query it with SQL.
4. Choose between batch, streaming, hybrid, and orchestrated pipelines.

## 1) Data Engineering Ecosystem

### Analytics (turn raw data into decisions)
- Business Intelligence dashboards
- Product and usage metrics
- Marketing and SEO analytics

### Data Science (build predictive systems)
- Recommendation systems
- Fraud detection
- Machine learning and LLM workflows

### Streaming systems (process events in real time)
- Social media feeds
- Video/event telemetry
- Monitoring and alerting pipelines

## 2) Core Data Engineering Lifecycle

1. **Data Sourcing**: discover and ingest data from internal/external systems.
2. **Data Storage**: store data in the right system and format.
3. **Data Processing**: clean, transform, aggregate, and enrich data.
4. **Data Orchestration**: schedule and coordinate multi-step workflows.

### Typical tools
- Sourcing: Python scripts, Airbyte, Fivetran
- Storage: Cloud Storage, databases, data lakes, data warehouses
- Processing: BigQuery, Spark, Beam, Dataflow
- Orchestration: Airflow/Cloud Composer, Prefect, Kestra

## 3) GCP Storage and Analytics Services

### Google Cloud Storage (GCS)
- Object storage (files/blobs inside buckets).
- Best for raw files, backups, archives, media, and data lake layers.
- Common classes: Standard, Nearline, Coldline, Archive.

### BigQuery
- Fully managed data warehouse for analytics.
- SQL over large structured/semi-structured datasets.
- Excellent for BI dashboards, reporting, and ML feature preparation.

### Datastore / Firestore
- NoSQL document-oriented storage for app workloads.
- Optimized for low-latency app reads/writes (not heavy analytics).

## 4) BigQuery vs Cloud Storage

| Feature | BigQuery | Cloud Storage |
|---|---|---|
| Service type | Data warehouse | Object storage |
| Main purpose | Analytics and SQL | Durable file storage |
| Data model | Structured / semi-structured | Unstructured / semi-structured |
| Query capability | Yes (SQL) | No native SQL engine |
| Pricing focus | Storage + query compute | Storage + operations + egress |

**Simple rule:**
- Use **Cloud Storage** to store data.
- Use **BigQuery** to analyze data.

## 5) BigQuery Hands-on Setup
Run the next cell to create a BigQuery client and define common variables.

In [ ]:
from pathlib import Path
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

project_id = "global-grammar-432121-d7"
dataset_id = "cars_data"
client = bigquery.Client(project=project_id)

print(f"Client ready for project: {project_id}")

### Query a Table (with fallback if copied table is missing)

In [ ]:
preferred_table = "cars_data_copy"
fallback_table = "cars_data"

def table_exists(table_name: str) -> bool:
    try:
        client.get_table(f"{project_id}.{dataset_id}.{table_name}")
        return True
    except Exception:
        return False

selected_table = preferred_table if table_exists(preferred_table) else fallback_table
table_fqn = f"`{project_id}.{dataset_id}.{selected_table}`"

query = f"""
SELECT *
FROM {table_fqn}
LIMIT 20
"""

df = client.query(query, location="US").to_dataframe()
print(f"Reading table: {selected_table}")
display(df)

### Load local CSV into BigQuery (Housing.csv)

In [ ]:
csv_path = Path("Housing.csv")
target_table = "housing_data"
dataset_ref = f"{project_id}.{dataset_id}"
table_id = f"{project_id}.{dataset_id}.{target_table}"

if not csv_path.exists():
    raise FileNotFoundError(f"File not found: {csv_path}")

try:
    client.get_dataset(dataset_ref)
except NotFound:
    ds = bigquery.Dataset(dataset_ref)
    ds.location = "US"
    client.create_dataset(ds)
    print(f"Dataset created: {dataset_ref}")

load_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

with csv_path.open("rb") as f:
    load_job = client.load_table_from_file(
        f,
        table_id,
        job_config=load_config,
        location="US",
    )

load_job.result()
table = client.get_table(table_id)
print(f"Loaded table: {table_id}")
print(f"Rows: {table.num_rows} | Columns: {len(table.schema)}")

## 6) Pipeline Patterns

### Batch pipelines
- Process data in scheduled chunks (hourly/daily/weekly).
- Best for large-volume ETL where low latency is not required.

### Stream pipelines
- Process events continuously in near real time.
- Best for fraud detection, monitoring, and live analytics.

### Hybrid pipelines
- Combine historical batch processing with real-time stream updates.
- Best when both trend analysis and low-latency actions are needed.

## 7) Orchestrated Pipelines
- Use orchestrators (Airflow / Cloud Composer / Prefect) to schedule and monitor tasks.
- Keep orchestration logic thin: dependencies, retries, and scheduling only.
- Put heavy transformations in dedicated compute layers (BigQuery, Dataflow, Spark).

### Good practice
Design each task as idempotent and observable (logs, metrics, alerts).

## 8) Suggested Next Steps
1. Add profiling queries (`COUNT`, null checks, duplicates) for `housing_data`.
2. Build a batch workflow (CSV load + validation + summary table).
3. Add a stream example with Pub/Sub + Dataflow to compare with batch.

## 9) Dataproc:
- Managed Spark and Hadoop service for big data processing.
- Best for complex transformations, machine learning pipelines, and when you need more control over the processing
- Apache Spark, Hadoop, Hive, Pig, Presto, and more.

- Set up cluster: issue `gcloud dataproc clusters create` with appropriate parameters (e.g., machine type, number of workers, region).
- Configure nodes: master and workers with specific machine types and disk sizes.
- Customize cluster with initialization actions (e.g., install libraries, set up environment).
- Manage security with IAM roles and service accounts, and control access to data sources.

## 10) Spark core and RDDs:
- Spark Core is the underlying engine for Spark applications, providing distributed task scheduling and memory management.
- Drive program: define the main function, create a SparkContext, and execute transformations and actions on RDDs.
- RDDs (Resilient Distributed Datasets) are the fundamental data structure in Spark
- RDDs are immutable, distributed collections of objects that can be processed in parallel across a cluster.
- RDDs support transformations (e.g., `map`, `filter`, `reduceByKey`) and actions (e.g., `collect`, `count`, `saveAsTextFile`).
